# Day 6 — Titanic で特徴量エンジニアリング

**今日のゴール**
1. 本物の（＝列に意味がある）データで、**人間が特徴量を作る → 精度が上がる**を自分の手で体験する
2. ベースライン **GBDT 0.776** を、自作の特徴量で**超える**

🧱 = 足場（私が用意した）　🎯 = 本筋（あなたが書く）　✋ = 走らせる前に予想

> 問題: 乗客データから `survived`（1=生存 / 0=死亡）を当てる二値分類（Kaggle の登竜門）。

## 🧱 1. 道具とデータ

In [ ]:
import pandas as pd
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import GradientBoostingClassifier

df = pd.read_csv('../../data/titanic.csv')
print('データ:', df.shape)
df.head(3)

## 🧱 2. 最低限の前処理（足場）

決定木やGBDTは **数字しか食べられない＆穴(NaN)があると詰まる**。だから動かす前に:
- 文字 → 数字（`sex`: female/male→1/0、`embarked`: S/C/Q→0/1/2）
- 穴を埋める（`age`/`fare` は中央値、`embarked` は最頻値）

> ここは今日の本筋（特徴量づくり）ではないので渡す。`name` `ticket` `cabin` はこの後の特徴量づくりで使うので、いったん置いておく。

In [ ]:
df['sex']      = (df['sex'] == 'female').astype(int)
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0]).map({'S':0, 'C':1, 'Q':2})
df['age']      = df['age'].fillna(df['age'].median())
df['fare']     = df['fare'].fillna(df['fare'].median())
print('前処理OK（NaN残り:', int(df[['sex','embarked','age','fare']].isna().sum().sum()), '）')

## 🧱 3. ベースライン（生の列だけ・特徴量づくり前）

これが「超える目標」。

In [ ]:
feats = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']
X, y = df[feats], df['survived']

base = cross_val_score(GradientBoostingClassifier(random_state=42), X, y, cv=10)
print(f'ベースライン GBDT : {base.mean():.3f} +/- {base.std():.3f}   <- これを超えたい')

## ✋ 予想する

どの特徴量を作れば効きそう？ 史実と列の意味から、作る前に2〜3個メモして。
（ヒント: 「子供は助かる」「一人ぼっち vs 家族連れ」「名前に隠れた肩書き」…）

## 🎯 4. ここが今日の本筋 — 特徴量を作って 0.776 を超える

**pandas で新しい列を作る基本:**
```python
df['新しい列名'] = (式)
```

**候補（自分で作ってみて）:**
- `family_size` = `sibsp` + `parch` + 1 … 自分を含めた同乗家族の人数
- `is_alone`   = `family_size == 1` … 一人ぼっちか
- `is_child`   = `age < 16` … 子供フラグ（"子供優先"を直接表す）

**やること:**
1. 上を参考に、`df` に新しい列を作る（`.astype(int)` で 1/0 にすると安全）
2. 作った列名を `feats` に足した新しい特徴量リストで `X2` を作る
3. ベースラインと**同じ GBDT・同じ cv=10** で測って、`0.776` を超えたか確認

In [ ]:
# 🎯 ここを自分で埋める

# 1. 特徴量を作る（例: df['family_size'] = df['sibsp'] + df['parch'] + 1）


# 2. 追加した列も入れた新しい特徴量リスト
feats2 = feats + [ ]   # ← 作った列名をここに足す

# 3. 同じ GBDT・cv=10 で測る
X2 = df[feats2]
# new = cross_val_score(GradientBoostingClassifier(random_state=42), X2, y, cv=10)
# print(f'特徴量あり GBDT : {new.mean():.3f} +/- {new.std():.3f}   (ベースライン 0.776)')

## 🧱 5. メモ
- 超えられた？ どの特徴量が効いた？（1個ずつ足して試すと、効き目が分かる）
- 次の超定番: `name` から肩書き(Mr/Mrs/Miss/**Master**=幼い男の子)を抽出する特徴量。